# 🧬 The LangChain Blueprint: From Blocks to RAG
## Complete Demo Notebook — Azure OpenAI + Chroma Edition

This notebook accompanies the **"The LangChain Blueprint"** presentation and provides all the runnable code examples organized by phase:

| Phase | Topic | Cells |
|-------|-------|-------|
| **Setup** | Install & Configure Azure OpenAI | 1–2 |
| **Phase 01** | Core Blocks — Model I/O, Prompts, Parsers | 3–6 |
| **Phase 02** | LCEL — Pipe Operator, Passthrough, Parallel | 7–10 |
| **Phase 03** | Data Ingestion — Load, Chunk, Embed, Store (Chroma) | 11–15 |
| **Phase 04** | RAG Pipeline — Retrieve, Augment, Generate | 16–20 |

| **Assignment** | Mini Project — "Ask My Documents" | 23–29 |

> **Prerequisites:** An Azure OpenAI resource with deployed models for **chat** (e.g., `gpt-4o-mini`) and **embeddings** (e.g., `text-embedding-3-small`).
>
> **Vector Store:** This notebook uses **ChromaDB** — an open-source, lightweight vector database with built-in persistence.

---

## ⚙️ Setup

### Cell 1 — Install Dependencies

Run this cell once to install all required packages.

In [ ]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
%pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-core \
    langchain-chroma \
    chromadb \
    pypdf \
    python-dotenv \
    tiktoken \
    beautifulsoup4

### Cell 2 — Azure OpenAI Configuration

Fill in your Azure OpenAI credentials below. You can also load them from a `.env` file.

```
# .env file (optional)
AZURE_OPENAI_API_KEY=your-key-here
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-06-01
AZURE_OPENAI_CHAT_DEPLOYMENT=gpt-4o-mini
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-small
```

In [ ]:
# ============================================================
# CELL 2: Azure OpenAI Configuration
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file if it exists

# ── Fill these in OR set them as environment variables ──
AZURE_OPENAI_API_KEY            = os.getenv("AZURE_OPENAI_API_KEY", "your-api-key-here")
AZURE_OPENAI_ENDPOINT           = os.getenv("AZURE_OPENAI_ENDPOINT", "https://your-resource.openai.azure.com/")
AZURE_OPENAI_API_VERSION        = os.getenv("AZURE_OPENAI_API_VERSION", "2024-06-01")
AZURE_OPENAI_CHAT_DEPLOYMENT    = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")

# Set as env vars so LangChain picks them up automatically
os.environ["AZURE_OPENAI_API_KEY"]     = AZURE_OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"]    = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION

print("✅ Configuration loaded!")
print(f"   Endpoint:     {AZURE_OPENAI_ENDPOINT}")
print(f"   Chat Model:   {AZURE_OPENAI_CHAT_DEPLOYMENT}")
print(f"   Embeddings:   {AZURE_OPENAI_EMBEDDING_DEPLOYMENT}")
print(f"   Vector Store: ChromaDB")

---

## 🧱 Phase 01 — Core Blocks

### Cell 3 — Model I/O: Azure Chat Model (Slide 3)

Azure OpenAI uses the `AzureChatOpenAI` class. You specify a **deployment name** instead of a model name.

In [ ]:
# ============================================================
# CELL 3: Model I/O — Initializing an Azure Chat Model
# ============================================================
from langchain_openai import AzureChatOpenAI

# Initialize the Azure Chat Model
model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
    max_tokens=512,
)

# Quick test
response = model.invoke("What is LangChain in one sentence?")
print(f"Type:    {type(response).__name__}")
print(f"Content: {response.content}")

### Cell 4 — Prompt Templates with Variables (Slide 4)

Templates keep your prompt **logic** separate from your **data**. Variables inside `{braces}` are injected at runtime.

In [ ]:
# ============================================================
# CELL 4: Prompt Templates
# ============================================================
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Respond concisely."),
    ("human",  "Explain the concept of {topic} in 2-3 sentences.")
])

rendered = prompt.invoke({
    "role": "senior AI engineer",
    "topic": "vector embeddings"
})

for msg in rendered.messages:
    print(f"[{msg.type.upper()}] {msg.content}")
    print()

### Cell 5 — Output Parsers: Text → JSON (Slide 5)

Parsers convert raw LLM text into structured Python objects.

In [ ]:
# ============================================================
# CELL 5: Output Parsers
# ============================================================
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Pydantic-based parser for type-safe output
class ConceptSummary(BaseModel):
    name: str = Field(description="Name of the concept")
    summary: str = Field(description="A 1-2 sentence explanation")
    difficulty: str = Field(description="beginner, intermediate, or advanced")

pydantic_parser = PydanticOutputParser(pydantic_object=ConceptSummary)

print("── Format Instructions (injected into prompt) ──")
print(pydantic_parser.get_format_instructions()[:300], "...")

### Cell 6 — 🧪 Code: Basic Prompt + Model + Parser Chain (Slide 6)

In [ ]:
# ============================================================
# CELL 6: [CODE] Basic Prompt + Model + Parser
# ============================================================
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a technical writer. "
     "Return ONLY valid JSON with keys: name, summary, use_case. "
     "No markdown, no extra text."),
    ("human", "Describe the concept: {concept}")
])

# 2. Model
model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

# 3. Parser
parser = JsonOutputParser()

# 4. Chain with pipe operator
chain = prompt | model | parser

# 5. Invoke
result = chain.invoke({"concept": "vector embeddings"})

print("── Result (Python dict) ──")
for key, value in result.items():
    print(f"  {key}: {value}")

---

## ⚡ Phase 02 — LCEL (LangChain Expression Language)

### Cell 7 — The Pipe Operator `|` (Slide 7)

In [ ]:
# ============================================================
# CELL 7: The Pipe Operator + Streaming
# ============================================================
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.7,
)

# Prompt → Model → Parser
chain = (
    ChatPromptTemplate.from_template("Tell me a fun fact about {topic}.")
    | model
    | StrOutputParser()
)

result = chain.invoke({"topic": "neural networks"})
print(result)

# Stream output token by token
print("\n── Streaming ──")
for chunk in chain.stream({"topic": "transformers"}):
    print(chunk, end="", flush=True)
print()

### Cell 8 — RunnablePassthrough (Slide 8)

In [ ]:
# ============================================================
# CELL 8: RunnablePassthrough
# ============================================================
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

def fake_retriever(query: str) -> str:
    """Simulates document retrieval."""
    knowledge_base = {
        "LCEL": "LCEL stands for LangChain Expression Language. "
                "It uses the pipe operator to compose chains.",
        "RAG":  "RAG stands for Retrieval-Augmented Generation. "
                "It grounds LLM answers in retrieved documents.",
    }
    for key, value in knowledge_base.items():
        if key.lower() in query.lower():
            return value
    return "No relevant context found."

setup = RunnableParallel(
    context  = lambda x: fake_retriever(x["question"]),
    question = RunnablePassthrough()
)

result = setup.invoke({"question": "What is LCEL?"})
print("── RunnableParallel Output ──")
print(f"  context:  {result['context']}")
print(f"  question: {result['question']}")

### Cell 9 — RunnableParallel: Fan-Out Tasks (Slide 9)

In [ ]:
# ============================================================
# CELL 9: RunnableParallel — Multi-Branch Analysis
# ============================================================
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

summarize_chain = (
    ChatPromptTemplate.from_template("Summarize in exactly 2 sentences:\n\n{text}")
    | model | StrOutputParser()
)

keyword_chain = (
    ChatPromptTemplate.from_template("Extract 5 keywords as a comma-separated list:\n\n{text}")
    | model | StrOutputParser()
)

tone_chain = (
    ChatPromptTemplate.from_template("What is the tone? Reply with one word:\n\n{text}")
    | model | StrOutputParser()
)

parallel_chain = RunnableParallel(
    summary=summarize_chain, keywords=keyword_chain, tone=tone_chain,
)

sample_text = """
LangChain is a framework for developing applications powered by large language models.
It provides modular components for prompt management, model interaction, data retrieval,
and chain composition. The framework supports both Python and JavaScript, making it
accessible to a wide range of developers building AI-powered applications.
"""

result = parallel_chain.invoke({"text": sample_text})

print("── Parallel Analysis Results ──")
print(f"\n📝 Summary:\n   {result['summary']}")
print(f"\n🏷️  Keywords:\n   {result['keywords']}")
print(f"\n🎭 Tone:\n   {result['tone']}")

### Cell 10 — 🧪 Code: Complete LCEL Chain (Slide 10)

In [ ]:
# ============================================================
# CELL 10: [CODE] Complete LCEL Chain with Report Generation
# ============================================================
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

summarize = (
    ChatPromptTemplate.from_template("Summarize in 2 sentences:\n{text}")
    | model | StrOutputParser()
)
keywords = (
    ChatPromptTemplate.from_template("List 5 keywords from:\n{text}")
    | model | StrOutputParser()
)

report_prompt = ChatPromptTemplate.from_template(
    """Based on the following analysis, write a brief structured report:

Summary: {summary}
Keywords: {keywords}

Write 3-4 sentences synthesizing these findings."""
)

full_chain = (
    RunnableParallel(summary=summarize, keywords=keywords)
    | report_prompt | model | StrOutputParser()
)

sample = """Vector databases like Chroma and Pinecone store high-dimensional embeddings
for fast similarity search. They are essential for RAG pipelines, enabling semantic
retrieval of relevant documents. Modern vector stores support metadata filtering,
hybrid search combining keywords with vectors, and horizontal scaling."""

report = full_chain.invoke({"text": sample})
print("── Generated Report ──")
print(report)

---

## 💾 Phase 03 — Data Ingestion

### Cell 11 — Document Loaders (Slide 11)

LangChain normalizes all data sources into `Document` objects with `page_content` and `metadata`.

In [ ]:
# ============================================================
# CELL 11: Document Loaders
# ============================================================
from langchain_core.documents import Document

# Create demo documents (replace with PyPDFLoader / WebBaseLoader in production)
manual_docs = [
    Document(
        page_content="LCEL stands for LangChain Expression Language. "
                     "It uses the pipe operator (|) to compose chains of runnables.",
        metadata={"source": "langchain_docs", "page": 1}
    ),
    Document(
        page_content="RunnablePassthrough forwards input unchanged. "
                     "RunnableParallel runs multiple chains simultaneously.",
        metadata={"source": "langchain_docs", "page": 2}
    ),
    Document(
        page_content="Prompt templates parameterize prompts with variables using curly braces. "
                     "ChatPromptTemplate.from_messages() creates multi-message templates "
                     "with system, human, and AI roles.",
        metadata={"source": "langchain_docs", "page": 3}
    ),
    Document(
        page_content="Output parsers transform raw LLM text into structured data. "
                     "JsonOutputParser returns Python dicts, while PydanticOutputParser "
                     "returns validated Pydantic model instances.",
        metadata={"source": "langchain_docs", "page": 4}
    ),
    Document(
        page_content="A Retriever searches a vector store and returns relevant documents. "
                     "It takes a query string and returns a list of Document objects.",
        metadata={"source": "langchain_docs", "page": 5}
    ),
    Document(
        page_content="Text splitters like RecursiveCharacterTextSplitter break documents "
                     "into smaller chunks. chunk_size controls the maximum characters per "
                     "chunk, and chunk_overlap preserves context at boundaries.",
        metadata={"source": "langchain_docs", "page": 6}
    ),
    Document(
        page_content="Embeddings transform text into high-dimensional vectors. "
                     "Similar texts produce vectors that are close together in vector space, "
                     "enabling semantic similarity search.",
        metadata={"source": "langchain_docs", "page": 7}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) combines document retrieval with "
                     "LLM generation. The retrieved context is injected into the prompt "
                     "so the model answers using factual, grounded information.",
        metadata={"source": "langchain_docs", "page": 8}
    ),
    Document(
        page_content="ChromaDB is an open-source vector database optimized for AI applications. "
                     "It supports persistence to disk, metadata filtering, and integrates "
                     "natively with LangChain via the langchain-chroma package.",
        metadata={"source": "langchain_docs", "page": 9}
    ),
    Document(
        page_content="Vector stores like Chroma, FAISS, and Pinecone store embeddings "
                     "and support similarity search. You convert your vector store to "
                     "a retriever with .as_retriever().",
        metadata={"source": "langchain_docs", "page": 10}
    ),
]

print(f"✅ Created {len(manual_docs)} demo documents\n")

# ── Option B: Load from PDF (uncomment if you have a PDF) ──
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("your_document.pdf")
# pdf_docs = loader.load()

# ── Option C: Load from web (uncomment to try) ──
# from langchain_community.document_loaders import WebBaseLoader
# web_loader = WebBaseLoader("https://python.langchain.com/docs/get_started/introduction")
# web_docs = web_loader.load()

doc = manual_docs[0]
print(f"── Sample Document ──")
print(f"  page_content: {doc.page_content[:80]}...")
print(f"  metadata:     {doc.metadata}")

### Cell 12 — The Art of Chunking (Slide 12)

In [ ]:
# ============================================================
# CELL 12: Text Splitting / Chunking
# ============================================================
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(manual_docs)

print(f"📄 Original documents:  {len(manual_docs)}")
print(f"✂️  After splitting:     {len(chunks)} chunks")
print(f"📏 Chunk size target:   200 chars")
print(f"🔗 Overlap:             50 chars")

print("\n── First 5 Chunks ──")
for i, chunk in enumerate(chunks[:5]):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"    \"{chunk.page_content[:100]}...\"")
    print(f"    metadata: {chunk.metadata}")

### Cell 13 — Embeddings: Words → Vectors (Slide 13)

In [ ]:
# ============================================================
# CELL 13: Embeddings with Azure OpenAI
# ============================================================
from langchain_openai import AzureOpenAIEmbeddings
import numpy as np

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Embed a single query
query_vector = embeddings.embed_query("What is LCEL?")
print(f"── Single Query Embedding ──")
print(f"  Dimensions: {len(query_vector)}")
print(f"  First 5 values: {query_vector[:5]}")

# Compare similarity
texts = ["king", "queen", "apple"]
vectors = embeddings.embed_documents(texts)

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n── Semantic Similarity ──")
print(f"  'king' ↔ 'queen': {cosine_similarity(vectors[0], vectors[1]):.4f}  (high — similar)")
print(f"  'king' ↔ 'apple': {cosine_similarity(vectors[0], vectors[2]):.4f}  (low  — different)")

### Cell 14 — Vector Stores: ChromaDB (Slide 14)

Store embedded chunks in a **Chroma** vector database. Chroma auto-persists to disk and supports metadata filtering.

In [ ]:
# ============================================================
# CELL 14: Vector Store — ChromaDB
# ============================================================
from langchain_chroma import Chroma
from langchain_openai import AzureOpenAIEmbeddings
import shutil, os

# Clean up any previous run
PERSIST_DIR = "./chroma_db"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

# Initialize embeddings
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Create a Chroma vector store from chunks (auto-persists to disk)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="langchain_demo",
)

print(f"✅ Chroma vector store created!")
print(f"   Chunks stored:      {len(chunks)}")
print(f"   Persist directory:  {PERSIST_DIR}")
print(f"   Collection:         langchain_demo")

# Test similarity search
query = "How does the pipe operator work?"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"\n── Top 3 Results for: \"{query}\" ──")
for i, (doc, score) in enumerate(results):
    print(f"\n  Result {i+1} (distance: {score:.4f}):")
    print(f"    \"{doc.page_content[:120]}...\"")
    print(f"    source: {doc.metadata}")

### Cell 15 — 🧪 Code: Full Ingestion Pipeline with Chroma (Slide 15)

In [ ]:
# ============================================================
# CELL 15: [CODE] Complete Ingestion Pipeline (Chroma)
# ============================================================
from langchain_chroma import Chroma
from langchain_openai import AzureOpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
import shutil, os

# ── 1. LOAD ──
docs = manual_docs
print(f"📥 Step 1 — Loaded {len(docs)} documents")

# ── 2. SPLIT ──
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"✂️  Step 2 — Split into {len(chunks)} chunks")

# ── 3. EMBED + STORE (Chroma auto-persists) ──
PERSIST_DIR = "./chroma_db"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="langchain_demo",
)
print(f"🧮 Step 3 — Embedded and stored {len(chunks)} vectors in Chroma")
print(f"💾 Step 4 — Auto-persisted to '{PERSIST_DIR}/'")

# ── 4. VERIFY by reloading from disk ──
loaded_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings,
    collection_name="langchain_demo",
)
test_results = loaded_store.similarity_search("What is RAG?", k=2)
print(f"\n✅ Verification — Reloaded store returned {len(test_results)} results:")
for i, doc in enumerate(test_results):
    print(f"   {i+1}. \"{doc.page_content[:80]}...\"")

---

## 🔍 Phase 04 — The RAG Pipeline

### Cell 16 — What is RAG? The Three Steps (Slide 16)

In [ ]:
# ============================================================
# CELL 16: RAG Concept — The Three Steps Visualized
# ============================================================

question = "What is LCEL and how does it work?"

# ── Step 1: RETRIEVE ──
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
retrieved_docs = retriever.invoke(question)

print("── Step 1: RETRIEVE ──")
print(f"   Query: \"{question}\"")
print(f"   Found {len(retrieved_docs)} relevant chunks:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"   [{i+1}] \"{doc.page_content[:100]}...\"")

# ── Step 2: AUGMENT ──
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
augmented_prompt = f"""Answer the question based ONLY on the following context:

{context}

Question: {question}
Answer:"""

print(f"\n── Step 2: AUGMENT ──")
print(f"   Prompt length: {len(augmented_prompt)} characters")
print(f"   Context chunks injected: {len(retrieved_docs)}")

# ── Step 3: GENERATE ──
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

answer = model.invoke(augmented_prompt)
print(f"\n── Step 3: GENERATE ──")
print(f"   {answer.content}")

### Cell 17 — The Retriever: Similarity vs MMR (Slide 17)

In [ ]:
# ============================================================
# CELL 17: Retriever Configuration & Testing
# ============================================================

# ── Similarity retriever ──
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

test_queries = [
    "What is RAG?",
    "How do output parsers work?",
    "Explain vector stores and Chroma",
    "What is chunking?",
]

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\n🔎 Query: \"{query}\"")
    print(f"   Top result: \"{docs[0].page_content[:90]}...\"")

# ── MMR retriever (balances relevance + diversity) ──
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6}
)

print("\n── MMR Retriever (diversity-optimized) ──")
mmr_docs = mmr_retriever.invoke("How does LangChain work?")
for i, doc in enumerate(mmr_docs):
    print(f"  [{i+1}] \"{doc.page_content[:90]}...\"")

### Cell 18 — Context Injection: The RAG Prompt (Slide 18)

In [ ]:
# ============================================================
# CELL 18: Context Injection — The RAG Prompt
# ============================================================
from langchain.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful AI assistant. Answer the user's question using "
     "ONLY the provided context. If the context doesn't contain the "
     "answer, say 'I don't have enough information to answer that.' "
     "Be concise and cite which part of the context supports your answer."),
    ("human",
     """Context:
{context}

Question: {question}"""
    )
])

# Demonstrate rendering
sample_context = "LCEL uses the pipe operator (|) to compose chains of runnables."
rendered = rag_prompt.invoke({
    "context": sample_context,
    "question": "What operator does LCEL use?"
})

print("── Rendered RAG Prompt ──")
for msg in rendered.messages:
    print(f"\n[{msg.type.upper()}]")
    print(f"  {msg.content}")

### Cell 19 — 🧪 Code: The Final Modular RAG Chain (Slide 19)

Everything comes together: **Chroma Retriever** + **Passthrough** + **Prompt** + **Model** + **Parser** — all connected with LCEL pipes.

In [ ]:
# ============================================================
# CELL 19: [CODE] The Final Modular RAG Chain (Chroma)
# ============================================================
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_chroma import Chroma

# ── 1. Load the persisted Chroma vector store ──
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
)
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="langchain_demo",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# ── 2. Model ──
model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

# ── 3. RAG Prompt ──
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using ONLY the provided context. Be concise and accurate. "
     "If the context doesn't contain the answer, say so."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# ── 4. Format docs helper ──
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── 5. The Complete RAG Chain ──
rag_chain = (
    RunnableParallel(
        context  = retriever | format_docs,
        question = RunnablePassthrough()
    )
    | rag_prompt
    | model
    | StrOutputParser()
)

# ── 6. Test it! ──
print("=" * 60)
print("  🧬 THE LANGCHAIN RAG CHAIN IS LIVE! (Chroma backend)")
print("=" * 60)

test_questions = [
    "What is LCEL?",
    "How do output parsers work?",
    "What is RAG and why is it useful?",
    "Explain how text chunking works.",
    "What is ChromaDB?",
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"\n❓ {q}")
    print(f"💡 {answer}")
    print("-" * 50)

### Cell 20 — 🎯 Interactive Q&A Session

Change the question and re-run!

In [ ]:
# ============================================================
# CELL 20: Interactive Q&A
# ============================================================

YOUR_QUESTION = "What is the difference between RunnablePassthrough and RunnableParallel?"

answer = rag_chain.invoke(YOUR_QUESTION)

print(f"❓ Question: {YOUR_QUESTION}")
print(f"\n💡 Answer:\n{answer}")

# Show retrieved sources
print("\n📚 Retrieved Sources:")
retrieved = retriever.invoke(YOUR_QUESTION)
for i, doc in enumerate(retrieved):
    print(f"  [{i+1}] (page {doc.metadata.get('page', '?')}) \"{doc.page_content[:100]}...\"")

---

## 🎁 Bonus — Streaming RAG + Conversation History

### Cell 21 — Streaming RAG Chain

In [ ]:
# ============================================================
# CELL 21: [BONUS] Streaming RAG Chain
# ============================================================

question = "Explain the complete RAG pipeline step by step."

print(f"❓ {question}\n")
print("💡 ", end="")

for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)

print("\n")

### Cell 22 — RAG with Conversation History

In [ ]:
# ============================================================
# CELL 22: [BONUS] RAG with Conversation History
# ============================================================
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

rag_with_history_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using ONLY the provided context. Be concise. "
     "Use the conversation history for follow-up questions."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_with_history = (
    RunnableParallel(
        context      = retriever | format_docs,
        question     = RunnablePassthrough(),
        chat_history = lambda x: x.get("chat_history", []),
    )
    | rag_with_history_prompt
    | model
    | StrOutputParser()
)

# Multi-turn conversation
chat_history = []

conversations = [
    "What is LCEL?",
    "How does it use the pipe operator?",
    "What about RunnableParallel?",
]

for question in conversations:
    answer = rag_with_history.invoke({
        "input": question,
        "question": question,
        "chat_history": chat_history,
    })

    print(f"\n👤 You:   {question}")
    print(f"🤖 Agent: {answer}")

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

print(f"\n{'=' * 50}")
print(f"📝 Conversation history: {len(chat_history)} messages")